# Train the Cipher Detective classifier on a **free** Colab GPU

No paid Hugging Face Space needed. This notebook trains the character-level
cipher classifier on Colab's free T4 GPU and uploads it to the Hub.

## Before you start (2 minutes)
1. **Turn on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.
2. **Get a free Hugging Face token:** sign in at <https://huggingface.co>, then
   go to **Settings → Access Tokens → Create new token**, choose type **Write**,
   and copy it. You'll paste it into the login cell below.

Then run the cells top to bottom (Shift+Enter). Total time: ~30–60 min, $0.

## 1. Confirm the GPU is on

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU! Do Runtime → Change runtime type → T4 GPU, then Runtime → Restart.'
print('GPU ready:', torch.cuda.get_device_name(0))

## 2. Get the code and the training data (LFS)

In [ ]:
!git clone --depth 1 https://github.com/systemslibrarian/cipher-detective-ai.git
%cd cipher-detective-ai
!git lfs install && git lfs pull
!wc -l data/splits/train.jsonl data/splits/val.jsonl

## 3. Install the training libraries

In [ ]:
!pip -q install 'transformers>=4.44' datasets accelerate scikit-learn sentencepiece

## 4. Log in to Hugging Face
Paste the **Write** token you created. It is not saved in the notebook.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Train and upload the model

- `--char-level` is the key improvement: one token per character, the right
  fit for ciphertext (the app applies the same spacing at inference).
- Change `HUB_MODEL_ID` to **your** username if it isn't `systemslibrarian`.
- This trains the drop-in **81-class** classifier the Space already expects.
  To instead train the higher-accuracy 7-way family model, add `--family-labels`
  (note: the Space's Compare Mode expects the 81-class labels).

In [ ]:
HUB_MODEL_ID = 'systemslibrarian/cipher-detective-classifier'  # <-- your-username/model

!python scripts/train_transformer.py \
  --data data/splits/train.jsonl \
  --test-data data/splits/val.jsonl \
  --model distilbert-base-uncased \
  --char-level \
  --out cipher_model \
  --epochs 5 --batch-size 32 --max-length 256 \
  --push-to-hub --hub-model-id $HUB_MODEL_ID

## 6. Point the Space at your model

In your Space's **Settings → Variables and secrets**, add:

```
CIPHER_MODEL_ID = systemslibrarian/cipher-detective-classifier
```

Restart the Space. The **About / Model Status** tab will confirm it loaded; if
anything fails, the app falls back to the heuristic by design. Done — no GPU
billing, because the Space only *runs* the model, it doesn't train it.